In [1]:
import boto3
import pandas as pd
import io
import cbbpy.mens_scraper as s

s3 = boto3.client('s3')


In [12]:
# Retrieve yesterday's games
obj = s3.get_object(Bucket='webstar-bucket', Key='todays_games.csv')
yesterdays_games = pd.read_csv(io.BytesIO(obj['Body'].read()))

# Retrieve results of yesterday's games
results = []

for i, game_id in enumerate(yesterdays_games['game_id'], start=1):
    try:
        game, _, _ = s.get_game(game_id, box = False, pbp = False)
        results.append(game)
        print(f'Successfully recieved {game_id}! - {i} / {len(yesterdays_games['game_id'])}')
    except Exception as e:
        print(f"Failed to retrieve {game_id}: {e}")

results = pd.concat(results, ignore_index=True)[['game_id', 'home_score', 'away_score', 'game_day']]

Successfully recieved 401808288! - 1 / 92
Successfully recieved 401851325! - 2 / 92
Successfully recieved 401851250! - 3 / 92
Successfully recieved 401851626! - 4 / 92
Successfully recieved 401823806! - 5 / 92
Successfully recieved 401823186! - 6 / 92
Successfully recieved 401822976! - 7 / 92
Successfully recieved 401809122! - 8 / 92
Successfully recieved 401856426! - 9 / 92
Successfully recieved 401851582! - 10 / 92
Successfully recieved 401808287! - 11 / 92
Successfully recieved 401805216! - 12 / 92
Successfully recieved 401851464! - 13 / 92
Successfully recieved 401851753! - 14 / 92
Successfully recieved 401851534! - 15 / 92
Successfully recieved 401825566! - 16 / 92
Successfully recieved 401825562! - 17 / 92
Successfully recieved 401809104! - 18 / 92
Successfully recieved 401805215! - 19 / 92
Successfully recieved 401851703! - 20 / 92
Successfully recieved 401820788! - 21 / 92
Successfully recieved 401822973! - 22 / 92
Successfully recieved 401808281! - 23 / 92
Successfully recieve

In [13]:
# Merge results onto yesterday's games
results['game_id'] = results['game_id'].astype('Int64')
yesterdays_games = pd.merge(yesterdays_games, results, on='game_id')

In [17]:
yesterdays_games['home_dff'] = yesterdays_games['home_score'] - yesterdays_games['away_score']
yesterdays_games['home_error'] = yesterdays_games['home_dff'] - (yesterdays_games['home_team_spread'] * -1)
yesterdays_games['home_error_abs'] = abs(yesterdays_games['home_error'])

# Concat onto existing archive
obj = s3.get_object(Bucket='webstar-bucket', Key='archived_results.csv')
archived_results = pd.read_csv(io.BytesIO(obj['Body'].read()))
archived_results = pd.concat([archived_results, yesterdays_games], ignore_index=True)

C:\Users\rpwju\AppData\Local\Temp\ipykernel_16348\573769447.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  archived_results = pd.concat([archived_results, yesterdays_games], ignore_index=True)
